# 2. Matroidal strata of the $E_6$-Grassmannian

This notebook follows Section 3 of the extended abstract. It covers

1. the equations of $E_6/P_6\subseteq\mathbb{P}^{26}$ and the charts $U_y\cong\mathbb{A}^{16}$ (Theorem 3.1),
2. supports of points and matroidal strata (Proposition 3.2, Definition 3.3),
3. the equations of a stratum in a chart, exchange graphs and leaf orders,
4. the main theorem (Theorem 3.5) and the table of dimensions (Table 1),
5. the two special orbits $Q_1$ and $Q_{\mathrm{nr}}$,
6. the failure of the frontier condition (Remark 3.7).

The functions are defined in `src/cayley_plane.jl` and `src/strata.jl`.

In [2]:
include("../src/E6P6CoxeterMatroids.jl");

## Equations

The coordinate ring `R` has one variable for each line, with the same name; `abc` is the list of the $27$ variables, so `abc[Q]` turns an index set into a list of coordinates. The $E_6$-invariant cubic is

$$D=\mathbf{a}^{T}C\,\mathbf{b}+\mathrm{Pf}(C),$$

where $C$ is the skew-symmetric matrix of the $c_{ij}$. It has one monomial for each of the $45$ tritangent planes.

In [3]:
D

a1*b2*c12 + a1*b3*c13 + a1*b4*c14 + a1*b5*c15 + a1*b6*c16 - a2*b1*c12 + a2*b3*c23 + a2*b4*c24 + a2*b5*c25 + a2*b6*c26 - a3*b1*c13 - a3*b2*c23 + a3*b4*c34 + a3*b5*c35 + a3*b6*c36 - a4*b1*c14 - a4*b2*c24 - a4*b3*c34 + a4*b5*c45 + a4*b6*c46 - a5*b1*c15 - a5*b2*c25 - a5*b3*c35 - a5*b4*c45 + a5*b6*c56 - a6*b1*c16 - a6*b2*c26 - a6*b3*c36 - a6*b4*c46 - a6*b5*c56 + c12*c34*c56 - c12*c35*c46 + c12*c36*c45 - c13*c24*c56 + c13*c25*c46 - c13*c26*c45 + c14*c23*c56 - c14*c25*c36 + c14*c26*c35 - c15*c23*c46 + c15*c24*c36 - c15*c26*c34 + c16*c23*c45 - c16*c24*c35 + c16*c25*c34

In [4]:
@show Set(findall(!iszero, e) for e in exponents(D)) == Set(tritangent_planes);

Set((findall(!iszero, e) for e = exponents(D))) == Set(tritangent_planes) = true


The $E_6$-Grassmannian is cut out by the $27$ partial derivatives $D_y$; they generate the ideal `I`. The quadric $D_y$ is a signed sum over the $5$ tritangent planes through $y$ (equation (2)).

In [5]:
Dy = gens(I)
@show Dy[6]                               # D_y for y = a6
@show all(Set(findall(!iszero, e) for e in exponents(Dy[y])) == Set(tritangent_pairs[y]) for y in 1:27)
@show dim(I);                             # 17 = dim(E6/P6) + 1

Dy[6] = -b1*c16 - b2*c26 - b3*c36 - b4*c46 - b5*c56
all((Set((findall(!iszero, e) for e = exponents(Dy[y]))) == Set(tritangent_pairs[y]) for y = 1:27)) = true
dim(I) = 17


## The big cell (Theorem 3.1)

Fix a line $y$. In the chart $U_y=\{p_y\ne 0\}$ we set $p_y=1$. Each quadric $D_{v^*}$, $v\in\lozenge(y)$, then reads $\pm(p_v-g_v)$ where $g_v$ is a quadric in the $16$ coordinates of $N(y)$. The function `isolate_vars(y)` returns the dictionary $v\mapsto g_v$.

In [6]:
g = isolate_vars(a6)

Dict{Any, Any} with 10 entries:
  c16 => -a2*c12 - a3*c13 - a4*c14 - a5*c15
  c36 => a1*c13 + a2*c23 - a4*c34 - a5*c35
  c56 => a1*c15 + a2*c25 + a3*c35 + a4*c45
  b3  => a3*b6 + c12*c45 - c14*c25 + c15*c24
  c26 => a1*c12 - a3*c23 - a4*c24 - a5*c25
  b4  => a4*b6 - c12*c35 + c13*c25 - c15*c23
  b1  => a1*b6 + c23*c45 - c24*c35 + c25*c34
  b5  => a5*b6 + c12*c34 - c13*c24 + c14*c23
  b2  => a2*b6 - c13*c45 + c14*c35 - c15*c34
  c46 => a1*c14 + a2*c24 + a3*c34 - a5*c45

Theorem 3.1 says that $p_y=1$, $p_v=g_v$ parametrizes $U_y$ by $\mathbb{A}^{16}$. It remains to see that the other $17$ quadrics vanish after this substitution, which `verify_big_cell` checks. We do it in all $27$ charts.

In [7]:
@show all(verify_big_cell, abc);

all(verify_big_cell, abc) = true


## Supports are matroids (Proposition 3.2)

The support of a point is the set of its nonzero coordinates. To illustrate the proposition we take random points of the chart $U_{a_6}$ with many zero coordinates and test their supports.

In [8]:
y = 6
function random_support()
    vals = [i in neighbours(y) ? QQ(rand(-1:1)) : QQ(0) for i in 1:27]
    p = [i == y ? QQ(1) : haskey(g, abc[i]) ? evaluate(g[abc[i]], vals) : vals[i] for i in 1:27]
    return findall(!iszero, p)
end

supports = unique(random_support() for _ in 1:2000)
@show length(supports)
@show all(is_E6P6_matroid, supports);

length(supports) = 1979
all(is_E6P6_matroid, supports) = true


## The equations of a stratum

Let $Q$ be a matroid and $y\in Q$. The stratum $\mathcal{M}_Q$ lies in the chart $U_y$. Setting the coordinates outside $Q$ to zero turns $g_v$ into $f_v$, and

$$\mathcal{M}_Q\cong\{p\in\mathbb{G}_m^{Q\cap N(y)} : f_v(p)=0\ (v\in E_y),\quad f_v(p)\neq 0\ (v\in\lozenge(y)\cap Q)\}.$$

`E6P6_realization_space(abc[Q], abc[y])` returns the ideal of the equations and the list of polynomials that must be invertible.

In [10]:
Q = E6_mats[111]
println(line_labels[Q])
y = good_chart(Q)
@show line_labels[y]
IQ, SQ = E6P6_realization_space(abc[Q], abc[y])
@show gens(IQ)
@show SQ;

["a1", "a2", "a3", "a4", "a5", "b1", "b2", "b3", "b4", "b6", "c12", "c13", "c24", "c35", "c36", "c45", "c46"]
line_labels[y] = "c12"
gens(IQ) = QQMPolyRingElem[-a5*b6 + c13*c24, -a3*b4 + a4*b3]
SQ = QQMPolyRingElem[a5, b3, a3, b6, b4, a4, c13, c24]


Each $f_v$ is a signed sum of monomials $p_up_{u^*}$ over pairwise disjoint pairs (a matching), and never a single monomial. The *exchange graph* is the union of these matchings, one for each equation; `exchange_graph(Q, y)` computes it from the incidence relation alone. A *leaf order* is an ordering of the matchings in which each has a vertex of degree one in the union of itself and its successors. Along a leaf order the equations can be solved one variable at a time, without leaving the torus.

In [11]:
matchings = exchange_graph(Q, y)
for m in matchings
    println([line_labels[p] for p in m])
end
@show leaf_order(matchings);

[["a5", "b6"], ["c13", "c24"]]
[["a3", "b4"], ["a4", "b3"]]
leaf_order(matchings) = [1, 2]


## The main theorem (Theorem 3.5)

`good_chart(Q)` returns a chart with a leaf order and as few equations as possible. Exactly one orbit has no such chart: the orbit of $Q_1$.

In [12]:
no_leaf_order = [i for i in 1:230 if isnothing(good_chart(E6_mats[i]))]
@show no_leaf_order
@show E6_mats[no_leaf_order[1]] == minimal_rep(Q1);

no_leaf_order = [116]
E6_mats[no_leaf_order[1]] == minimal_rep(Q1) = true


In the chart $y=c_{45}$, the three equations of $Q_1$ are the $2\times 2$ minors of a $2\times 3$ matrix. They cut out a translate of a subtorus of codimension $2$, so $\mathcal{M}_{Q_1}$ is still a torus, of dimension $8$.

In [13]:
IQ1, _ = E6P6_realization_space(abc[Q1], c45)
gens(IQ1)

3-element Vector{QQMPolyRingElem}:
 -a2*b3 + a3*b2
 a1*b3 - a3*b1
 -a1*b2 + a2*b1

Exactly one orbit is not realizable: the orbit of $Q_{\mathrm{nr}}=Q_1\cup\{c_{14}\}$, number $139$ in our list. In the chart at $a_1$ its stratum requires $f_{c_{15}}=f_{c_{16}}=0$ while $p_{a_4}$ and $f_{c_{14}}$ are invertible. The linear syzygy (3) for $w=b_1$ gives $p_{a_4}f_{c_{14}}+p_{a_5}f_{c_{15}}+p_{a_6}f_{c_{16}}=0$, a contradiction.

In [14]:
println(line_labels[Qnr])
@show minimal_rep(Qnr) == E6_mats[139]

restrict = hom(R, R, [x in abc[Qnr] ? x : R(0) for x in abc])     # set the coordinates outside Qnr to zero
f = Dict(v => restrict(gv) for (v, gv) in isolate_vars(a1))
@show f[c14]
@show f[c15]
@show f[c16]
@show a4*f[c14] + a5*f[c15] + a6*f[c16];

["a1", "a2", "a3", "a4", "a5", "a6", "b1", "b2", "b3", "b4", "b5", "b6", "c12", "c13", "c14", "c23", "c45", "c46", "c56"]
minimal_rep(Qnr) == E6_mats[139] = true
f[c14] = a5*c45 + a6*c46
f[c15] = -a4*c45 + a6*c56
f[c16] = -a4*c46 - a5*c56
a4 * f[c14] + a5 * f[c15] + a6 * f[c16] = 0


`stratum_is_nonempty(Q)` decides realizability over $\overline{\mathbb{Q}}$ by saturating the ideal of equations with respect to the units. Only orbit $139$ fails.

In [15]:
@time [i for i in 1:230 if !stratum_is_nonempty(E6_mats[i])]

  5.387625 seconds (29.12 M allocations: 1.557 GiB, 12.51% gc time, 84.61% compilation time)


1-element Vector{Int64}:
 139

For the other $229$ orbits, $\mathcal{M}_Q$ is an open subscheme of a torus of dimension $|Q\cap N(y)|-|E_y|$. This gives Table 1.

In [16]:
realizable = [Q for Q in E6_mats if Q != E6_mats[139]]
dims = stratum_dimension.(realizable)
[d => count(==(d), dims) for d in 0:16]

17-element Vector{Pair{Int64, Int64}}:
  0 => 1
  1 => 1
  2 => 2
  3 => 3
  4 => 8
  5 => 12
  6 => 23
  7 => 34
  8 => 43
  9 => 38
 10 => 30
 11 => 17
 12 => 9
 13 => 4
 14 => 2
 15 => 1
 16 => 1

Finally, every nonempty stratum has points over $\overline{\mathbb{F}}_p$ for every prime $p$: a Gröbner basis over $\mathbb{Z}$ of the stratum in a chart has only $\pm1$ as leading coefficients, so it remains a Gröbner basis of a proper ideal modulo every prime. `realizable_over_every_prime(Q)` performs this test. The script `scripts/strata_table.jl` runs it on all orbits and writes `data/strata.csv`; here we look at a few.

In [17]:
@show realizable_over_every_prime(Q1)
@show all(realizable_over_every_prime, realizable[1:40:end]);

realizable_over_every_prime(Q1) = true
all(realizable_over_every_prime, realizable[1:40:end]) = true


Realizability over a fixed finite field is more delicate (Remark 3.6). For instance $\mathcal{L}$ is not realizable over $\mathbb{F}_2$: all coordinates would equal $1$, and each $D_y$ has an odd number of terms $\pm1$.

In [18]:
@show all(isodd(length(collect(terms(f)))) for f in Dy);

all((isodd(length(collect(terms(f)))) for f = Dy)) = true


## The strata do not form a stratification (Remark 3.7)

Let $Q=\{a_1,\dots,a_5,b_6,c_{12},\dots,c_{15},c_{23},c_{45}\}$ and $P=\{a_2,\dots,a_5,c_{12},\dots,c_{15}\}\subseteq Q$. Both strata have dimension $6$, and the closure of $\mathcal{M}_Q$ meets $\mathcal{M}_P$ without containing it.

`E6P6_stratum(Q)` computes the ideal of the closure of $\mathcal{M}_Q$ in $\mathbb{P}^{26}$ by saturation.

In [19]:
Q = [a1, a2, a3, a4, a5, b6, c12, c13, c14, c15, c23, c45]
P = [a2, a3, a4, a5, c12, c13, c14, c15]
index(Q) = [position_abc[x] for x in Q]

@show is_E6P6_matroid(index(Q)), is_E6P6_matroid(index(P))
@show stratum_dimension(index(Q)), stratum_dimension(index(P))
@show test_Q2_meets_Q1bar(Q, P)
@show frontier_condition_holds(Q, P);

(is_E6P6_matroid(index(Q)), is_E6P6_matroid(index(P))) = (true, true)
(stratum_dimension(index(Q)), stratum_dimension(index(P))) = (6, 6)
test_Q2_meets_Q1bar(Q, P) = true
frontier_condition_holds(Q, P) = false
